<a href="https://colab.research.google.com/github/Krishna101010101010/Samudrika-Underwater-Image-Enhancement-for-Maritime-Security/blob/main/Samudrika_Yolov11_Dataset_Merger_and_Training_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Samudrika-v1 — YOLO Dataset Merger & Class Remapper
=====================================================
Merges 5 Roboflow datasets into one unified YOLO dataset
with consistent class IDs.

Final class schema:
  0 = submarine
  1 = diver
  2 = naval_mine

Input datasets (all inside BASE_DIR):
  Naval mine.v1i.yolov11/          → naval_mine (drop rope)
  Naval mine 2.v1i.yolov11 3/      → naval_mine
  Underwater Mines.v1i.yolov11/    → naval_mine
  Shipdetection.v2-ship_8classes.yolov11/ → submarine only
  underwater divers.v1i.yolov11/   → diver
  ROV.v2i.yolov11/                 → SKIPPED (Ikan = fish)

Output structure:
  Samudrika_YOLO_dataset/
    images/
      train/
      val/
      test/
    labels/
      train/
      val/
      test/
    data.yaml
    merge_report.txt
"""

import os
import shutil
import random
import time
from pathlib import Path
from datetime import datetime


# ═══════════════════════════════════════════════════════════
# CONFIG — update BASE_DIR to your Drive path
# ═══════════════════════════════════════════════════════════
BASE_DIR   = Path('/content/drive/MyDrive/YOLO.v11 Dataset/Yolo Dataset')
OUTPUT_DIR = Path('/content/Samudrika_YOLO_Merged-dataset')

RANDOM_SEED  = 42
TRAIN_RATIO  = 0.80
VAL_RATIO    = 0.10
# TEST_RATIO  = 0.10  (remainder)

VALID_IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# ═══════════════════════════════════════════════════════════
# CLASS REMAPPING TABLE
#
# Format:
#   'dataset_folder_name': {
#       old_class_id: new_class_id,   # -1 = drop this class
#   }
# ═══════════════════════════════════════════════════════════
CLASS_NAMES = ['submarine', 'diver', 'naval_mine']

REMAP = {
    'Naval mine.v1i.yolov11': {
        0: 2,   # Bottom-Mine  → naval_mine
        1: 2,   # Moored-Mine  → naval_mine
        2: -1,  # rope         → DROP
    },
    'Naval mine 2.v1i.yolov11 3': {
        0: 2,   # Bottom-Mine  → naval_mine
        1: 2,   # Moored-Mine  → naval_mine
    },
    'Underwater Mines.v1i.yolov11': {
        0: 2,   # Mines        → naval_mine
    },
    'Shipdetection.v2-ship_8classes.yolov11': {
        0: -1,  # CoastGuard      → DROP
        1: -1,  # aircraft carrier→ DROP
        2: -1,  # container       → DROP
        3: -1,  # cruise          → DROP
        4: -1,  # fish-b          → DROP
        5: -1,  # sail boat       → DROP
        6:  0,  # submarine       → submarine ✓
        7: -1,  # warship         → DROP
    },
    'underwater divers.v1i.yolov11': {
        0: 1,   # divers       → diver
    },
    # ROV.v2i.yolov11 → SKIPPED (Ikan = fish, wrong dataset)
}

# Roboflow split folder names → our split names
SPLIT_MAP = {
    'train': 'train',
    'valid': 'val',
    'test' : 'test',
}


# ═══════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════

def fmt_time(seconds):
    if seconds < 60:
        return f'{seconds:.1f}s'
    elif seconds < 3600:
        return f'{int(seconds//60)}m {int(seconds%60)}s'
    else:
        return f'{int(seconds//3600)}h {int((seconds%3600)//60)}m'


def is_image(path: Path) -> bool:
    return path.suffix.lower() in VALID_IMG_EXTS


def remap_label_file(src: Path, dst: Path, id_map: dict) -> tuple[int, int]:
    """
    Read a YOLO .txt label file, remap class IDs, write to dst.
    Returns (lines_kept, lines_dropped).
    """
    if not src.exists():
        # Empty label file (background image) — write empty
        dst.write_text('')
        return 0, 0

    lines_in  = src.read_text().strip().splitlines()
    lines_out = []
    dropped   = 0

    for line in lines_in:
        line = line.strip()
        if not line:
            continue
        parts  = line.split()
        old_id = int(parts[0])
        new_id = id_map.get(old_id, -1)
        if new_id == -1:
            dropped += 1
            continue
        lines_out.append(f'{new_id} {" ".join(parts[1:])}')

    dst.write_text('\n'.join(lines_out) + ('\n' if lines_out else ''))
    return len(lines_out), dropped


def find_label(img_path: Path, dataset_root: Path, split: str) -> Path:
    """
    Given an image path, find its corresponding .txt label file.
    Roboflow structure: images/ and labels/ are siblings under train|valid|test
    """
    label_path = dataset_root / split / 'labels' / (img_path.stem + '.txt')
    return label_path


def count_dataset_images(ds_dir: Path) -> int:
    """Count total images across all splits in a dataset folder."""
    total = 0
    for rf_split in SPLIT_MAP:
        img_dir = ds_dir / rf_split / 'images'
        if img_dir.exists():
            total += len([f for f in img_dir.iterdir() if is_image(f)])
    return total


def process_dataset(
    ds_name: str,
    ds_dir: Path,
    id_map: dict,
    out_images: dict,
    out_labels: dict,
    report: list,
    global_counter: dict   # {'done': int, 'total': int, 'start': float}
) -> dict:
    """
    Process one dataset: copy images + remapped labels for all splits.
    Returns stats dict.
    """
    stats = {s: {'images': 0, 'labels_kept': 0, 'labels_dropped': 0}
             for s in ['train', 'val', 'test']}

    prefix = ds_name.replace(' ', '_').replace('.', '_')[:20]

    for rf_split, our_split in SPLIT_MAP.items():
        img_dir   = ds_dir / rf_split / 'images'
        label_dir = ds_dir / rf_split / 'labels'

        if not img_dir.exists():
            continue

        images = sorted([f for f in img_dir.iterdir() if is_image(f)])

        for i, img_path in enumerate(images):
            # Unique filename: prefix_originalname to avoid collisions
            new_stem = f'{prefix}_{img_path.stem}'
            new_img_name   = new_stem + img_path.suffix
            new_label_name = new_stem + '.txt'

            # Copy image
            shutil.copy2(img_path, out_images[our_split] / new_img_name)

            # Remap and copy label
            src_label = label_dir / (img_path.stem + '.txt')
            dst_label = out_labels[our_split] / new_label_name
            kept, dropped = remap_label_file(src_label, dst_label, id_map)

            stats[our_split]['images']        += 1
            stats[our_split]['labels_kept']   += kept
            stats[our_split]['labels_dropped']+= dropped

            # ── Live ETA update every 100 images ──
            global_counter['done'] += 1
            done  = global_counter['done']
            total = global_counter['total']
            if done % 100 == 0 or done == total:
                elapsed   = time.time() - global_counter['start']
                rate      = done / elapsed if elapsed > 0 else 0
                remaining = (total - done) / rate if rate > 0 else 0
                pct       = done / total * 100
                bar_fill  = int(pct / 2.5)
                bar       = '█' * bar_fill + '░' * (40 - bar_fill)
                finish_at = datetime.fromtimestamp(
                    time.time() + remaining
                ).strftime('%H:%M:%S')
                print(
                    f'\r  [{bar}] {pct:5.1f}% '
                    f'{done}/{total} imgs | '
                    f'{rate:.1f} img/s | '
                    f'ETA {fmt_time(remaining)} (done ~{finish_at})',
                    end='', flush=True
                )

    return stats


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════

def main():
    SCRIPT_START = time.time()

    print('=' * 65)
    print('  Samudrika-v1 — YOLO Dataset Merger & Class Remapper')
    print(f'  Started : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print('=' * 65)

    # ── Create output dirs ──
    out_images = {}
    out_labels = {}
    for split in ['train', 'val', 'test']:
        p_img = OUTPUT_DIR / 'images' / split
        p_lbl = OUTPUT_DIR / 'labels' / split
        p_img.mkdir(parents=True, exist_ok=True)
        p_lbl.mkdir(parents=True, exist_ok=True)
        out_images[split] = p_img
        out_labels[split] = p_lbl

    report_lines = [
        'Samudrika-v1 YOLO Merge Report',
        f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
        f'Output    : {OUTPUT_DIR}',
        '=' * 60,
        '',
        f'Final class schema:',
        f'  0 = submarine',
        f'  1 = diver',
        f'  2 = naval_mine',
        '',
    ]

    total_stats = {s: {'images': 0, 'labels_kept': 0, 'labels_dropped': 0}
                   for s in ['train', 'val', 'test']}

    # ── Pre-scan: count total images across all datasets for ETA ──
    print('\n[PRE-SCAN] Counting total images...')
    total_image_count = 0
    valid_datasets    = {}
    for ds_name in REMAP:
        ds_dir = BASE_DIR / ds_name
        if ds_dir.exists():
            n = count_dataset_images(ds_dir)
            valid_datasets[ds_name] = n
            total_image_count += n
            print(f'  {ds_name[:45]:45s} : {n:5d} images')
        else:
            valid_datasets[ds_name] = 0
            print(f'  {ds_name[:45]:45s} : NOT FOUND — will skip')

    print(f'\n  Total to process : {total_image_count} images')
    print(f'  Estimated time   : {fmt_time(total_image_count / 8.0)} '
          f'(assuming ~8 img/s on Drive)')
    finish_est = datetime.fromtimestamp(
        time.time() + total_image_count / 8.0
    ).strftime('%H:%M:%S')
    print(f'  Estimated finish : ~{finish_est}')
    print()

    # ── Global counter shared across all datasets for live ETA ──
    global_counter = {
        'done' : 0,
        'total': total_image_count,
        'start': time.time(),
    }

    # ── Process each dataset ──
    for ds_name, id_map in REMAP.items():
        ds_dir = BASE_DIR / ds_name

        print(f'\n[{ds_name}]')

        if not ds_dir.exists():
            print(f'  [WARN] Folder not found — skipping')
            report_lines.append(f'[{ds_name}] — SKIPPED (folder not found)')
            continue

        t0    = time.time()
        stats = process_dataset(
            ds_name, ds_dir, id_map,
            out_images, out_labels, report_lines,
            global_counter
        )
        print()  # newline after progress bar
        elapsed = time.time() - t0

        for split in ['train', 'val', 'test']:
            s = stats[split]
            if s['images'] > 0:
                print(f'  {split:5s} : {s["images"]:4d} images | '
                      f'{s["labels_kept"]:5d} annotations kept | '
                      f'{s["labels_dropped"]:4d} dropped')
            total_stats[split]['images']        += s['images']
            total_stats[split]['labels_kept']   += s['labels_kept']
            total_stats[split]['labels_dropped']+= s['labels_dropped']

        print(f'  Done in {fmt_time(elapsed)}')

        report_lines += [
            f'[{ds_name}]',
            f'  train : {stats["train"]["images"]} images | {stats["train"]["labels_kept"]} annotations',
            f'  val   : {stats["val"]["images"]} images | {stats["val"]["labels_kept"]} annotations',
            f'  test  : {stats["test"]["images"]} images | {stats["test"]["labels_kept"]} annotations',
            '',
        ]

    # ── Verification ──
    print(f'\n[VERIFY]')
    for split in ['train', 'val', 'test']:
        img_count = len(list(out_images[split].iterdir()))
        lbl_count = len(list(out_labels[split].iterdir()))
        match     = '✓' if img_count == lbl_count else '✗ MISMATCH'
        print(f'  {split:5s} : {img_count} images · {lbl_count} labels {match}')

    # ── Class distribution check ──
    print(f'\n[CLASS DISTRIBUTION]')
    class_counts = {i: 0 for i in range(len(CLASS_NAMES))}
    for split in ['train', 'val', 'test']:
        for lbl_file in out_labels[split].iterdir():
            for line in lbl_file.read_text().strip().splitlines():
                if line.strip():
                    cls = int(line.split()[0])
                    class_counts[cls] = class_counts.get(cls, 0) + 1

    for cls_id, cls_name in enumerate(CLASS_NAMES):
        count = class_counts.get(cls_id, 0)
        bar   = '█' * min(40, count // 100)
        print(f'  {cls_id} {cls_name:15s} : {count:6d} annotations  {bar}')

    # ── Write data.yaml ──
    yaml_content = f"""# Samudrika-v1 YOLO Dataset
# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
# Classes: {len(CLASS_NAMES)}

path: {OUTPUT_DIR}
train: images/train
val:   images/val
test:  images/test

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
    (OUTPUT_DIR / 'data.yaml').write_text(yaml_content)
    print(f'\n[data.yaml] written ✓')

    # ── Summary ──
    total_imgs = sum(total_stats[s]['images'] for s in ['train', 'val', 'test'])
    total_anns = sum(total_stats[s]['labels_kept'] for s in ['train', 'val', 'test'])
    total_drop = sum(total_stats[s]['labels_dropped'] for s in ['train', 'val', 'test'])
    total_time = time.time() - SCRIPT_START

    report_lines += [
        'MERGE SUMMARY',
        f'  Total images     : {total_imgs}',
        f'  Total annotations: {total_anns}',
        f'  Annotations dropped (wrong class): {total_drop}',
        '',
        'CLASS DISTRIBUTION',
    ] + [f'  {i} {CLASS_NAMES[i]:15s} : {class_counts.get(i,0)} annotations'
         for i in range(len(CLASS_NAMES))] + [
        '',
        f'TIMING',
        f'  Total time : {fmt_time(total_time)}',
    ]

    (OUTPUT_DIR / 'merge_report.txt').write_text('\n'.join(report_lines))

    print(f'\n{"=" * 65}')
    print(f'  DONE')
    print(f'  Total images     : {total_imgs}')
    print(f'  Total annotations: {total_anns}')
    print(f'  Annotations dropped: {total_drop}')
    print(f'  Total time       : {fmt_time(total_time)}')
    print(f'  Output           : {OUTPUT_DIR}')
    print(f'{"=" * 65}')


if __name__ == '__main__':
    main()

  Samudrika-v1 — YOLO Dataset Merger & Class Remapper
  Started : 2026-05-24 22:38:12

[PRE-SCAN] Counting total images...
  Naval mine.v1i.yolov11                        :   597 images
  Naval mine 2.v1i.yolov11 3                    :  1093 images
  Underwater Mines.v1i.yolov11                  :   116 images
  Shipdetection.v2-ship_8classes.yolov11        :  5865 images
  underwater divers.v1i.yolov11                 :  2156 images

  Total to process : 9827 images
  Estimated time   : 20m 28s (assuming ~8 img/s on Drive)
  Estimated finish : ~22:58:59


[Naval mine.v1i.yolov11]
  [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   5.1% 500/9827 imgs | 5.1 img/s | ETA 30m 35s (done ~23:10:44)
  train :  522 images |   734 annotations kept |  483 dropped
  val   :   51 images |    70 annotations kept |   46 dropped
  test  :   24 images |    33 annotations kept |   20 dropped
  Done in 3m 36s

[Naval mine 2.v1i.yolov11 3]
  [██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  16.3% 1600/9827 imgs | 2